# Cox proportional hazards baseline: time to transplant

**Question:** from what we know at initial listing, how quickly will a candidate receive a kidney transplant?

**Endpoint (cause-specific hazard of transplant):**

| `outcome` | Treatment |
| -- | -- |
| `transplanted` | **Event** (`event_transplant = 1`) |
| `still_waiting` | Right-censored at follow-up cutoff |
| `died`, `removed_too_sick`, `removed_administrative`, `transplanted_elsewhere` | Competing events, **censored at removal** (cause-specific Cox) |
| `unknown` | Excluded (unresolved outcome) |

Censoring competing events gives the *cause-specific hazard* of transplant, not the probability of
transplant. Survival curves from this model therefore overstate transplant probability when there are competing risks;
a Fine-Gray / Aalen-Johansen model is the next comparison step (see the project overview).

**Other decisions:**
- The 59 records with `END_DATE < INIT_DATE` (missing `days_to_event`) are excluded.
- The 790 same-day records (`days_to_event == 0`) are set to 0.5 days so every event has positive time.
- Temporal split: listed 2015–2021 = train, 2022 onward = test. All imputation and scaling is fit on train only.
- `INIT_CPRA` is complete through 2022 but about 80% missing for listings from 2023 onward (an extract artifact,
  not a clinical pattern). The training years never have it missing, so a missingness indicator cannot be learned.
  The test set is therefore split into **2022** (cPRA complete, primary test) and **2023+** (cPRA mostly missing),
  and a second Cox model without cPRA is evaluated on 2023+.
- Features: the listing-time set from `data/data_dictionary.md`; no outcome or identifier fields.



In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc
from sksurv.util import Surv

pd.set_option("display.width", 140)

In [2]:
# Look for the restricted extract in the documented location first
candidates = [
    Path("../data/restricted/kidney_waitlist_analytic.csv.gz"),
    Path("../../data/restricted/kidney_waitlist_analytic.csv.gz"),
    Path("data/restricted/kidney_waitlist_analytic.csv.gz"),
    Path("../../data/kidney_waitlist_analytic.csv.gz"),
    Path("repo/src/data/kidney_waitlist_analytic.csv.gz"),
]
DATA_PATH = next(p for p in candidates if p.exists())
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
raw_df.shape

(494862, 38)

## 1. Build the survival cohort

In [3]:
cohort = raw_df[(raw_df["outcome"] != "unknown") & raw_df["days_to_event"].notna()].copy()

cohort["duration"] = cohort["days_to_event"].astype(float).clip(lower=0.5)
cohort["event"] = (cohort["outcome"] == "transplanted").astype("int8")
cohort["INIT_DATE"] = pd.to_datetime(cohort["INIT_DATE"], errors="coerce")
cohort["DIALYSIS_DATE"] = pd.to_datetime(cohort["DIALYSIS_DATE"], errors="coerce")

print(f"excluded rows: {len(raw_df) - len(cohort):,}")
cohort["outcome"].value_counts().to_frame("rows")

excluded rows: 4,874


,rows
outcome,
transplanted,234429
still_waiting,103300
removed_administrative,45331
transplanted_elsewhere,37243
removed_too_sick,35219
died,34466


## 2. Listing-time features

In [4]:
def decode_functional(codes):
    codes = pd.to_numeric(codes, errors="coerce")
    scale = pd.Series("other_unresolved", index=codes.index)
    scale[codes.isna()] = "missing"
    scale[codes.eq(996)] = "not_applicable"
    scale[codes.eq(998)] = "unknown"
    scale[codes.eq(1)] = "adl_independent"
    adult = codes.between(2010, 2100)
    peds = codes.between(4010, 4100)
    scale[adult] = "adult_karnofsky"
    scale[peds] = "pediatric_lansky"
    percent = pd.Series(np.nan, index=codes.index)
    percent[adult] = codes[adult] - 2000
    percent[peds] = codes[peds] - 4000
    return scale, percent


feat = pd.DataFrame(index=cohort.index)
feat["INIT_AGE"] = cohort["INIT_AGE"].astype(float)
feat["BMI_TCR"] = cohort["BMI_TCR"].where(cohort["BMI_TCR"].between(10, 80))
feat["bmi_missing"] = feat["BMI_TCR"].isna().astype("int8")
feat["INIT_CPRA"] = cohort["INIT_CPRA"].astype(float)
cpra_missing = feat["INIT_CPRA"].isna()  # kept for evaluation only; constant (never missing) in train
feat["functional_scale"], feat["functional_percent"] = decode_functional(cohort["FUNC_STAT_TCR"])
feat["dialysis_years"] = (
    (cohort["INIT_DATE"] - cohort["DIALYSIS_DATE"]).dt.days.clip(lower=0).fillna(0) / 365.25
)
feat["ON_DIALYSIS"] = cohort["ON_DIALYSIS"]
feat["GENDER"] = cohort["GENDER"]
feat["ABO"] = cohort["ABO"]
feat["A2A2B_ELIGIBILITY"] = cohort["A2A2B_ELIGIBILITY"]
feat["INIT_STAT"] = cohort["INIT_STAT"].replace({4050: 4010, 4060: 4010, 4020: 4010})  # rare codes pooled with active
feat["ETHCAT"] = cohort["ETHCAT"]
feat["REGION"] = cohort["REGION"]

numeric_cols = ["INIT_AGE", "BMI_TCR", "INIT_CPRA", "functional_percent", "dialysis_years"]
categorical_cols = [
    "functional_scale", "ON_DIALYSIS", "GENDER", "ABO",
    "A2A2B_ELIGIBILITY", "INIT_STAT", "ETHCAT", "REGION",
]
for col in categorical_cols:
    feat[col] = feat[col].astype("string").fillna("unknown")

## 3. Temporal train/test split, preprocessing fit on train only

In [5]:
# cPRA missingness by listing year (aggregate) motivates splitting the test period
print(cpra_missing.groupby(cohort["INIT_DATE"].dt.year).mean().round(3).to_frame("cpra_missing_rate").T)

INIT_DATE          2015  2016  2017  2018  2019  2020  2021  2022   2023   2024   2025   2026
cpra_missing_rate   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.795  0.829  0.812  0.808


In [6]:
is_train = cohort["INIT_DATE"] < "2022-01-01"
is_test22 = cohort["INIT_DATE"].between("2022-01-01", "2022-12-31")
is_test23 = cohort["INIT_DATE"] >= "2023-01-01"
train_idx = cohort.index[is_train]

medians = feat.loc[train_idx, numeric_cols].median()
feat[numeric_cols] = feat[numeric_cols].fillna(medians)

X = pd.get_dummies(feat, columns=categorical_cols, drop_first=True, dtype="float32")
# Drop dummy levels too rare in train to estimate stably
rare = [c for c in X.columns if c not in numeric_cols and X.loc[train_idx, c].sum() < 50]
X = X.drop(columns=rare)

means = X.loc[train_idx, numeric_cols].mean()
stds = X.loc[train_idx, numeric_cols].std()
X[numeric_cols] = (X[numeric_cols] - means) / stds

data = X.assign(duration=cohort["duration"], event=cohort["event"])
train = data[is_train]
test22 = data[is_test22]
test23 = data[is_test23]
test23_cpra_observed = data[is_test23 & ~cpra_missing]

print(f"dropped rare levels: {rare}")
splits = {
    "train (2015–2021)": train,
    "test 2022 (cPRA complete)": test22,
    "test 2023+ (all)": test23,
    "test 2023+ (cPRA observed)": test23_cpra_observed,
}
pd.DataFrame({
    name: {
        "rows": len(df),
        "events": int(df["event"].sum()),
        "event_rate": round(df["event"].mean(), 3),
        "median_days": df["duration"].median(),
    }
    for name, df in splits.items()
}).T

dropped rare levels: ['ABO_A2B', 'ETHCAT_998']


,rows,events,event_rate,median_days
train (2015–2021),270979.0,148684.0,0.549,623.0
test 2022 (cPRA complete),44003.0,22733.0,0.517,504.0
test 2023+ (all),175006.0,63012.0,0.360,268.0
test 2023+ (cPRA observed),33008.0,12045.0,0.365,261.5


## 4. Fit the Cox model

A small L2 penalty keeps one-hot coefficients stable without changing the ranking much.

In [7]:
cph = CoxPHFitter(penalizer=0.01)
cph.fit(train, duration_col="duration", event_col="event")

hr = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
hr.sort_values("exp(coef)").round(3)

,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,p
covariate,,,,
functional_scale_missing,0.462,0.210,1.015,0.054
ON_DIALYSIS_Y,0.644,0.636,0.653,0.000
ABO_B,0.659,0.646,0.674,0.000
INIT_STAT_4099,0.674,0.665,0.682,0.000
ABO_O,0.705,0.697,0.713,0.000
ETHCAT_2,0.724,0.715,0.734,0.000
ETHCAT_7,0.733,0.679,0.792,0.000
ETHCAT_5,0.734,0.718,0.749,0.000
ETHCAT_4,0.818,0.806,0.831,0.000


Hazard ratio > 1 means a **faster** transplant (higher hazard of the event); < 1 means a longer wait.
Numeric features are standardized, so their HR is per 1 SD.

## 5. Evaluate with the C-index

- **Harrell's C**: the share of comparable pairs where the candidate with the higher predicted hazard is transplanted first.
- **Uno's C** (IPCW): corrects for the heavy censoring in recent listings; truncated at 2 years because the 2023+ window is short.

Each test period is evaluated separately. **Test 2022** is the primary result because its features are complete.
The project target is C-index >= 0.72.

In [8]:
def evaluate(model, train_df, test_df, tau=2 * 365):
    risk_test = model.predict_partial_hazard(test_df).to_numpy()
    y_train = Surv.from_arrays(train_df["event"].astype(bool), train_df["duration"])
    y_test = Surv.from_arrays(test_df["event"].astype(bool), test_df["duration"])
    times = np.array([180, 365, 730])
    auc, _ = cumulative_dynamic_auc(y_train, y_test, risk_test, times)
    return {
        "rows": len(test_df),
        "Harrell C": concordance_index(test_df["duration"], -risk_test, test_df["event"]),
        "Uno C (tau=2y)": concordance_index_ipcw(y_train, y_test, risk_test, tau=tau)[0],
        "AUC(t) 6mo": auc[0],
        "AUC(t) 1y": auc[1],
        "AUC(t) 2y": auc[2],
    }


def evaluate_all(model, frame=None, drop=()):
    frame = (data if frame is None else frame).drop(columns=list(drop))
    tr = frame[is_train]
    return {
        "train (2015–2021)": {"rows": len(tr), "Harrell C": model.concordance_index_},
        "test 2022 (cPRA complete)": evaluate(model, tr, frame[is_test22]),
        "test 2023+ (all)": evaluate(model, tr, frame[is_test23]),
        "test 2023+ (cPRA observed)": evaluate(model, tr, frame[is_test23 & ~cpra_missing]),
    }


def show(results):
    return pd.concat(
        {name: pd.DataFrame(r).T for name, r in results.items()}, names=["model", "split"]
    ).round(4)


results = {"Cox, all listing features": evaluate_all(cph)}
show(results)

rows  Harrell C  Uno C (tau=2y)  AUC(t) 6mo  AUC(t) 1y  AUC(t) 2y
model                     split                                                                                            
Cox, all listing features train (2015–2021)           270979.0     0.6714             NaN         NaN        NaN        NaN
                          test 2022 (cPRA complete)    44003.0     0.6717          0.6734      0.7262     0.7206     0.7175
                          test 2023+ (all)            175006.0     0.6542          0.6524      0.6828     0.6860     0.7024
                          test 2023+ (cPRA observed)   33008.0     0.6371          0.6354      0.6618     0.6632     0.6761

## 6. Cox without cPRA for listings where cPRA is unavailable

Imputing the training median for ~80% of 2023+ candidates erases the cPRA signal and shifts their predicted risk.
A model fit without cPRA is the honest option for those rows; in deployment, use the full model when cPRA is recorded
and this one when it is not.

In [9]:
cph_nocpra = CoxPHFitter(penalizer=0.01)
cph_nocpra.fit(train.drop(columns=["INIT_CPRA"]), duration_col="duration", event_col="event")
results["Cox, no cPRA"] = evaluate_all(cph_nocpra, drop=["INIT_CPRA"])

# Route each 2023+ candidate to the full model when cPRA is observed, otherwise to the no-cPRA model.
# Partial hazards from two models are on different scales, so compare them through 2-year transplant probability.
def transplant_prob(model, df, t=730):
    return 1 - model.predict_survival_function(df, times=[t]).iloc[0].to_numpy()

observed = ~cpra_missing[test23.index].to_numpy()
risk_routed = np.where(
    observed,
    transplant_prob(cph, test23),
    transplant_prob(cph_nocpra, test23.drop(columns=["INIT_CPRA"])),
)
print(
    "Harrell C, test 2023+ with routed models:",
    round(concordance_index(test23["duration"], -risk_routed, test23["event"]), 4),
)
show(results)

Harrell C, test 2023+ with routed models: 0.6537


rows  Harrell C  Uno C (tau=2y)  AUC(t) 6mo  AUC(t) 1y  AUC(t) 2y
model                     split                                                                                            
Cox, all listing features train (2015–2021)           270979.0     0.6714             NaN         NaN        NaN        NaN
                          test 2022 (cPRA complete)    44003.0     0.6717          0.6734      0.7262     0.7206     0.7175
                          test 2023+ (all)            175006.0     0.6542          0.6524      0.6828     0.6860     0.7024
                          test 2023+ (cPRA observed)   33008.0     0.6371          0.6354      0.6618     0.6632     0.6761
Cox, no cPRA              train (2015–2021)           270979.0     0.6717             NaN         NaN        NaN        NaN
                          test 2022 (cPRA complete)    44003.0     0.6719          0.6737      0.7259     0.7211     0.7192
                          test 2023+ (all)            175006.0     0.6554          0.6536      0.6841     0.6883     0.7057
                          test 2023+ (cPRA observed)   33008.0     0.6444          0.6428      0.6701     0.6747     0.6922

## 7. Nonlinear Cox: clinical bins for cPRA, age, BMI, function and dialysis time

A single linear cPRA term is nearly flat (HR ~0.94 per SD) because the true effect is not monotone: under current
allocation policy, candidates with cPRA 98–100 receive strong priority, while highly sensitized candidates below that
threshold wait longer. Age, BMI and functional status also failed the PH test, which is often a sign of a wrong
functional form. Fixed clinical cut points are used (not quantiles), so nothing is learned from the test period.

cPRA that is missing (2023+ only) falls into the training median's bin; read the **cPRA observed** rows for a fair test.

In [10]:
binned = pd.DataFrame(index=feat.index)
binned["cpra_bin"] = pd.cut(feat["INIT_CPRA"], [-np.inf, 0, 79, 97, np.inf], labels=["0", "1-79", "80-97", "98-100"])
binned["age_bin"] = pd.cut(feat["INIT_AGE"], [-np.inf, 17, 34, 49, 64, 74, np.inf],
                           labels=["<18", "18-34", "35-49", "50-64", "65-74", "75+"])
binned["bmi_bin"] = pd.cut(feat["BMI_TCR"], [-np.inf, 18.5, 25, 30, 35, 40, np.inf],
                           labels=["<18.5", "18.5-25", "25-30", "30-35", "35-40", "40+"])
binned["function_bin"] = pd.cut(feat["functional_percent"], [-np.inf, 40, 60, 80, np.inf],
                                labels=["<=40", "50-60", "70-80", "90-100"])
binned["dialysis_bin"] = pd.cut(feat["dialysis_years"], [-np.inf, 0, 1, 3, 5, np.inf],
                                labels=["none", "<1y", "1-3y", "3-5y", "5y+"])
bin_cols = list(binned.columns)
reference = {"cpra_bin": "0", "age_bin": "50-64", "bmi_bin": "25-30", "function_bin": "90-100", "dialysis_bin": "none"}

X_nl = pd.concat([binned, feat[categorical_cols + ["bmi_missing"]]], axis=1)
for col in bin_cols:
    X_nl[col] = X_nl[col].astype("string")
X_nl = pd.get_dummies(X_nl, columns=bin_cols + categorical_cols, dtype="float32")
# Drop the chosen reference level for binned features, the first level for the rest
drop_ref = [f"{col}_{level}" for col, level in reference.items()]
drop_first = [next(c for c in X_nl.columns if c.startswith(f"{col}_")) for col in categorical_cols]
X_nl = X_nl.drop(columns=drop_ref + drop_first)
X_nl = X_nl.drop(columns=[c for c in X_nl.columns if X_nl.loc[train_idx, c].sum() < 50])
data_nl = X_nl.assign(duration=cohort["duration"], event=cohort["event"])

cph_nl = CoxPHFitter(penalizer=0.01)
cph_nl.fit(data_nl[is_train], duration_col="duration", event_col="event")
cph_nl.summary.loc[
    [c for c in cph_nl.summary.index if c.startswith(tuple(bin_cols))],
    ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"],
].round(3)

,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%
covariate,,,
cpra_bin_1-79,0.954,0.937,0.971
cpra_bin_80-97,0.965,0.931,1.000
cpra_bin_98-100,0.549,0.526,0.573
age_bin_18-34,1.433,1.410,1.457
age_bin_35-49,1.123,1.109,1.137
age_bin_65-74,0.993,0.978,1.008
age_bin_75+,1.159,1.105,1.215
age_bin_<18,1.995,1.759,2.264
bmi_bin_18.5-25,1.069,1.055,1.084


Hazard ratios are relative to the reference bins: cPRA 0, age 50–64, BMI 25–30, function 90–100, no dialysis.

In [11]:
results["Cox, binned nonlinear"] = evaluate_all(cph_nl, frame=data_nl)
show(results)

rows  Harrell C  Uno C (tau=2y)  AUC(t) 6mo  AUC(t) 1y  AUC(t) 2y
model                     split                                                                                            
Cox, all listing features train (2015–2021)           270979.0     0.6714             NaN         NaN        NaN        NaN
                          test 2022 (cPRA complete)    44003.0     0.6717          0.6734      0.7262     0.7206     0.7175
                          test 2023+ (all)            175006.0     0.6542          0.6524      0.6828     0.6860     0.7024
                          test 2023+ (cPRA observed)   33008.0     0.6371          0.6354      0.6618     0.6632     0.6761
Cox, no cPRA              train (2015–2021)           270979.0     0.6717             NaN         NaN        NaN        NaN
                          test 2022 (cPRA complete)    44003.0     0.6719          0.6737      0.7259     0.7211     0.7192
                          test 2023+ (all)            175006.0     0.6554          0.6536      0.6841     0.6883     0.7057
                          test 2023+ (cPRA observed)   33008.0     0.6444          0.6428      0.6701     0.6747     0.6922
Cox, binned nonlinear     train (2015–2021)           270979.0     0.6819             NaN         NaN        NaN        NaN
                          test 2022 (cPRA complete)    44003.0     0.6828          0.6851      0.7418     0.7363     0.7324
                          test 2023+ (all)            175006.0     0.6684          0.6663      0.7015     0.7051     0.7236
                          test 2023+ (cPRA observed)   33008.0     0.6491          0.6474      0.6766     0.6798     0.7004

## 8. Deceased-donor transplant endpoint

`transplanted` mixes deceased-donor and living-donor transplants. A living-donor transplant depends on whether a
relative or friend can donate, which listing-time clinical features barely capture, and it usually happens fast.
That noise hurts the ranking. The allocation question is about deceased-donor kidneys, so this endpoint counts only those:

| `REM_CD` | Treatment |
| -- | -- |
| `4`, `18` (deceased-donor, incl. emergency) | **Event** |
| `15` (living donor) | Competing event, censored at transplant |
| `43`, `45` (transplant, subtype unresolved) | Competing event, censored (do not guess the donor type) |
| everything else | Same as before |

`REM_CD` is used only to build the label, never as a feature.

In [12]:
rem_cd = pd.to_numeric(cohort["REM_CD"], errors="coerce")
event_deceased = rem_cd.isin([4, 18]).astype("int8")
print(
    pd.DataFrame({
        "deceased donor (4, 18)": rem_cd.isin([4, 18]).groupby(is_train).sum(),
        "living donor (15)": rem_cd.eq(15).groupby(is_train).sum(),
        "unresolved (43, 45)": rem_cd.isin([43, 45]).groupby(is_train).sum(),
    }).rename(index={True: "train", False: "test (2022+)"})
)

data_dd = data.assign(event=event_deceased)
data_nl_dd = data_nl.assign(event=event_deceased)

cph_dd = CoxPHFitter(penalizer=0.01)
cph_dd.fit(data_dd[is_train], duration_col="duration", event_col="event")
cph_nl_dd = CoxPHFitter(penalizer=0.01)
cph_nl_dd.fit(data_nl_dd[is_train], duration_col="duration", event_col="event")

results_dd = {
    "Cox linear, deceased-donor endpoint": evaluate_all(cph_dd, frame=data_dd),
    "Cox binned, deceased-donor endpoint": evaluate_all(cph_nl_dd, frame=data_nl_dd),
}
show(results_dd)

              deceased donor (4, 18)  living donor (15)  unresolved (43, 45)
INIT_DATE                                                                   
test (2022+)                   62569              22114                 1062
train                         104623              41628                 2433


rows  Harrell C  Uno C (tau=2y)  AUC(t) 6mo  AUC(t) 1y  AUC(t) 2y
model                               split                                                                                            
Cox linear, deceased-donor endpoint train (2015–2021)           270979.0     0.7170             NaN         NaN        NaN        NaN
                                    test 2022 (cPRA complete)    44003.0     0.7209          0.7154      0.7862     0.7674     0.7524
                                    test 2023+ (all)            175006.0     0.7162          0.7101      0.7481     0.7425     0.7505
                                    test 2023+ (cPRA observed)   33008.0     0.6891          0.6830      0.7167     0.7111     0.7202
Cox binned, deceased-donor endpoint train (2015–2021)           270979.0     0.7258             NaN         NaN        NaN        NaN
                                    test 2022 (cPRA complete)    44003.0     0.7276          0.7231      0.7925     0.7767     0.7634
                                    test 2023+ (all)            175006.0     0.7219          0.7164      0.7551     0.7524     0.7645
                                    test 2023+ (cPRA observed)   33008.0     0.6936          0.6880      0.7212     0.7193     0.7364

C-index values for different endpoints answer different questions, so compare them with that in mind: the
deceased-donor model ranks how fast someone gets a kidney through allocation, not any transplant.

In [13]:
cph_nl_dd.summary.loc[
    [c for c in cph_nl_dd.summary.index if c.startswith(tuple(bin_cols))],
    ["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"],
].round(3)

,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%
covariate,,,
cpra_bin_1-79,0.966,0.946,0.987
cpra_bin_80-97,1.115,1.073,1.158
cpra_bin_98-100,0.584,0.559,0.611
age_bin_18-34,1.193,1.169,1.217
age_bin_35-49,0.991,0.976,1.006
age_bin_65-74,1.116,1.096,1.135
age_bin_75+,1.240,1.169,1.314
age_bin_<18,2.118,1.843,2.435
bmi_bin_18.5-25,1.050,1.033,1.067


## 9. Sensitivity: drop sensitive attributes (race/ethnicity, sex, region)

The data dictionary asks for a comparison with and without these fields.

In [14]:
sensitive = [c for c in X.columns if c.startswith(("ETHCAT_", "GENDER_", "REGION_"))]
cph_ns = CoxPHFitter(penalizer=0.01)
cph_ns.fit(train.drop(columns=sensitive), duration_col="duration", event_col="event")
results["Cox, no ETHCAT/GENDER/REGION"] = evaluate_all(cph_ns, drop=sensitive)
show(results)

rows  Harrell C  Uno C (tau=2y)  AUC(t) 6mo  AUC(t) 1y  AUC(t) 2y
model                        split                                                                                            
Cox, all listing features    train (2015–2021)           270979.0     0.6714             NaN         NaN        NaN        NaN
                             test 2022 (cPRA complete)    44003.0     0.6717          0.6734      0.7262     0.7206     0.7175
                             test 2023+ (all)            175006.0     0.6542          0.6524      0.6828     0.6860     0.7024
                             test 2023+ (cPRA observed)   33008.0     0.6371          0.6354      0.6618     0.6632     0.6761
Cox, no cPRA                 train (2015–2021)           270979.0     0.6717             NaN         NaN        NaN        NaN
                             test 2022 (cPRA complete)    44003.0     0.6719          0.6737      0.7259     0.7211     0.7192
                             test 2023+ (all)            175006.0     0.6554          0.6536      0.6841     0.6883     0.7057
                             test 2023+ (cPRA observed)   33008.0     0.6444          0.6428      0.6701     0.6747     0.6922
Cox, binned nonlinear        train (2015–2021)           270979.0     0.6819             NaN         NaN        NaN        NaN
                             test 2022 (cPRA complete)    44003.0     0.6828          0.6851      0.7418     0.7363     0.7324
                             test 2023+ (all)            175006.0     0.6684          0.6663      0.7015     0.7051     0.7236
                             test 2023+ (cPRA observed)   33008.0     0.6491          0.6474      0.6766     0.6798     0.7004
Cox, no ETHCAT/GENDER/REGION train (2015–2021)           270979.0     0.6578             NaN         NaN        NaN        NaN
                             test 2022 (cPRA complete)    44003.0     0.6598          0.6621      0.7144     0.7040     0.6977
                             test 2023+ (all)            175006.0     0.6439          0.6417      0.6717     0.6703     0.6797
                             test 2023+ (cPRA observed)   33008.0     0.6356          0.6337      0.6613     0.6598     0.6667

## 10. Bootstrap confidence intervals for the test C-index

The best model's test C-index is close to the 0.72 target, so a point estimate is not enough.
The fitted models stay fixed; test candidates are resampled with replacement (200 draws), which measures uncertainty
from the finite test set. The paired difference uses the same resample for both models.

In [15]:
def bootstrap_cindex(df, risks, n_boot=200, seed=0):
    rng = np.random.default_rng(seed)
    t, e = df["duration"].to_numpy(), df["event"].to_numpy()
    draws = {name: [] for name in risks}
    for _ in range(n_boot):
        i = rng.integers(0, len(df), len(df))
        for name, r in risks.items():
            draws[name].append(concordance_index(t[i], -r[i], e[i]))
    return pd.DataFrame(draws)


ci_rows = []
for split_name, mask in {
    "test 2022 (cPRA complete)": is_test22,
    "test 2023+ (all)": is_test23,
}.items():
    risks = {
        "Cox binned, deceased-donor": cph_nl_dd.predict_partial_hazard(data_nl_dd[mask]).to_numpy(),
        "Cox linear, deceased-donor": cph_dd.predict_partial_hazard(data_dd[mask]).to_numpy(),
    }
    draws = bootstrap_cindex(data_nl_dd[mask], risks)
    draws["binned - linear"] = draws["Cox binned, deceased-donor"] - draws["Cox linear, deceased-donor"]
    for name in draws.columns:
        ci_rows.append({
            "split": split_name,
            "model": name,
            "mean": draws[name].mean(),
            "95% CI low": draws[name].quantile(0.025),
            "95% CI high": draws[name].quantile(0.975),
            "share of draws >= 0.72": (draws[name] >= 0.72).mean() if name != "binned - linear" else np.nan,
        })
pd.DataFrame(ci_rows).set_index(["split", "model"]).round(4)

mean  95% CI low  95% CI high  share of draws >= 0.72
split                     model                                                                              
test 2022 (cPRA complete) Cox binned, deceased-donor  0.7278      0.7242       0.7316                   1.000
                          Cox linear, deceased-donor  0.7211      0.7172       0.7245                   0.720
                          binned - linear             0.0067      0.0052       0.0082                     NaN
test 2023+ (all)          Cox binned, deceased-donor  0.7219      0.7195       0.7245                   0.925
                          Cox linear, deceased-donor  0.7162      0.7137       0.7186                   0.000
                          binned - linear             0.0057      0.0048       0.0066                     NaN

## 11. Subgroup C-index on the 2022 test set (fairness check)

Compares the original linear model (any transplant) with the best model (binned, deceased-donor endpoint).
Each model is scored against its own endpoint. Aggregate only; groups with fewer than 500 test rows are not shown.
The project's fairness target is a subgroup gap within 0.05.

In [16]:
test22_nl_dd = data_nl_dd[is_test22]
models_for_audit = {
    "linear, any transplant": (cph.predict_partial_hazard(test22).to_numpy(), test22),
    "binned, deceased-donor": (cph_nl_dd.predict_partial_hazard(test22_nl_dd).to_numpy(), test22_nl_dd),
}
groups = {
    "ETHCAT": cohort.loc[test22.index, "ETHCAT"],
    "GENDER": cohort.loc[test22.index, "GENDER"],
    "REGION": cohort.loc[test22.index, "REGION"],
}
rows = []
for name, values in groups.items():
    for level in values.dropna().unique():
        mask = (values == level).to_numpy()
        if mask.sum() < 500:
            continue
        row = {"attribute": name, "group": level, "rows": int(mask.sum())}
        for model_name, (risk, df) in models_for_audit.items():
            row[model_name] = concordance_index(
                df["duration"].to_numpy()[mask], -risk[mask], df["event"].to_numpy()[mask]
            )
        rows.append(row)
subgroups = pd.DataFrame(rows).sort_values(["attribute", "group"])
subgroups.round(4)

,attribute,group,rows,"linear, any transplant","binned, deceased-donor"
0,ETHCAT,1,17641,0.6459,0.7181
1,ETHCAT,2,13079,0.6887,0.7250
3,ETHCAT,4,8865,0.6804,0.7360
2,ETHCAT,5,3501,0.6836,0.7404
5,GENDER,F,16765,0.6637,0.7136
4,GENDER,M,27238,0.6767,0.7364
13,REGION,1,1818,0.6402,0.7069
6,REGION,2,5089,0.6523,0.7025
9,REGION,3,5961,0.6775,0.7197
15,REGION,4,5132,0.6828,0.7522


In [17]:
# Largest minus smallest subgroup C-index within each attribute
subgroups.groupby("attribute")[list(models_for_audit)].agg(lambda s: s.max() - s.min()).round(4).rename(
    columns=lambda c: f"gap, {c}"
)

,"gap, linear, any transplant","gap, binned, deceased-donor"
attribute,,
ETHCAT,0.0428,0.0223
GENDER,0.0130,0.0228
REGION,0.0522,0.0616


## 12. Proportional-hazards check for the best model

Schoenfeld-residual test (rank time transform) on a 20k-row training sample of the binned deceased-donor model.
With very large samples every test is "significant", so look at which terms have the largest test statistics
rather than at p-values alone. Terms near the top are candidates for stratification or time-varying effects.

In [18]:
from lifelines.statistics import proportional_hazard_test

sample = data_nl_dd[is_train].sample(20_000, random_state=0)
cph_sample = CoxPHFitter(penalizer=0.01).fit(sample, duration_col="duration", event_col="event")
ph = proportional_hazard_test(cph_sample, sample, time_transform="rank").summary
print(f"terms with p < 0.01: {(ph['p'] < 0.01).sum()} of {len(ph)}")
ph.sort_values("test_statistic", ascending=False).head(12).round(4)

terms with p < 0.01: 20 of 53


,test_statistic,p,-log2(p)
dialysis_bin_5y+,310.7328,0.0000,228.6165
ON_DIALYSIS_Y,119.9478,0.0000,90.3147
INIT_STAT_4099,119.2544,0.0000,89.8104
dialysis_bin_3-5y,98.8865,0.0000,74.9854
function_bin_<=40,73.1451,0.0000,56.2042
dialysis_bin_1-3y,51.4565,0.0000,40.3132
dialysis_bin_<1y,39.4776,0.0000,31.4888
A2A2B_ELIGIBILITY_1.0,24.3028,0.0000,20.2123
REGION_5,23.9696,0.0000,19.9626
age_bin_18-34,13.9770,0.0002,12.3997


## 13. Stratified Cox: separate baseline hazards by dialysis time

Dialysis duration has the worst PH violation: its advantage (accrued allocation waiting time) is concentrated early
after listing. Stratifying gives each dialysis group its own baseline hazard instead of forcing one constant HR.

A stratified model's partial hazard no longer contains the dialysis effect, so ranking by it would ignore dialysis
entirely. Candidates are instead ranked by **predicted probability of deceased-donor transplant within 2 years**,
which uses each stratum's own baseline. Variant B also stratifies on temporarily inactive status, the next-worst term.

In [19]:
data_st = data_nl_dd.drop(columns=[c for c in data_nl_dd.columns if c.startswith("dialysis_bin_")])
data_st["dialysis_stratum"] = binned["dialysis_bin"].astype(str)
data_st["inactive_stratum"] = data_st.pop("INIT_STAT_4099").astype(int)


def risk_at(model, df, t=730):
    # With strata, lifelines returns columns grouped by stratum, so realign to the input row order
    survival = model.predict_survival_function(df, times=[t]).iloc[0]
    return 1 - survival.reindex(df.index).to_numpy()


def evaluate_risk(risk_test, train_df, test_df, tau=2 * 365):
    y_train = Surv.from_arrays(train_df["event"].astype(bool), train_df["duration"])
    y_test = Surv.from_arrays(test_df["event"].astype(bool), test_df["duration"])
    auc, _ = cumulative_dynamic_auc(y_train, y_test, risk_test, np.array([180, 365, 730]))
    return {
        "rows": len(test_df),
        "Harrell C": concordance_index(test_df["duration"], -risk_test, test_df["event"]),
        "Uno C (tau=2y)": concordance_index_ipcw(y_train, y_test, risk_test, tau=tau)[0],
        "AUC(t) 6mo": auc[0],
        "AUC(t) 1y": auc[1],
        "AUC(t) 2y": auc[2],
    }


test_splits = {
    "test 2022 (cPRA complete)": is_test22,
    "test 2023+ (all)": is_test23,
    "test 2023+ (cPRA observed)": is_test23 & ~cpra_missing,
}
variants = {
    "A: strata dialysis": (["dialysis_stratum"], ["inactive_stratum"]),
    "B: strata dialysis + inactive": (["dialysis_stratum", "inactive_stratum"], []),
}
strat_models, results_st = {}, {}
for name, (strata, as_feature) in variants.items():
    cols = [c for c in data_st.columns if c not in ("dialysis_stratum", "inactive_stratum")] + strata + as_feature
    frame = data_st[cols]
    model = CoxPHFitter(penalizer=0.01)
    model.fit(frame[is_train], duration_col="duration", event_col="event", strata=strata)
    strat_models[name] = (model, frame)
    results_st[f"Cox stratified {name}"] = {
        split: evaluate_risk(risk_at(model, frame[mask]), frame[is_train], frame[mask])
        for split, mask in test_splits.items()
    }

# Unstratified best model, ranked the same way for a like-for-like comparison
results_st["Cox binned, deceased-donor (unstratified)"] = {
    split: evaluate_risk(risk_at(cph_nl_dd, data_nl_dd[mask]), data_nl_dd[is_train], data_nl_dd[mask])
    for split, mask in test_splits.items()
}
show(results_st)

rows  Harrell C  Uno C (tau=2y)  AUC(t) 6mo  AUC(t) 1y  \
model                                        split                                                                                    
Cox stratified A: strata dialysis            test 2022 (cPRA complete)    44003.0     0.7284          0.7245      0.7939     0.7791   
                                             test 2023+ (all)            175006.0     0.7229          0.7176      0.7568     0.7547   
                                             test 2023+ (cPRA observed)   33008.0     0.6984          0.6930      0.7271     0.7262   
Cox stratified B: strata dialysis + inactive test 2022 (cPRA complete)    44003.0     0.7298          0.7256      0.7964     0.7814   
                                             test 2023+ (all)            175006.0     0.7261          0.7206      0.7609     0.7585   
                                             test 2023+ (cPRA observed)   33008.0     0.7014          0.6961      0.7305     0.7305   
Cox binned, deceased-donor (unstratified)    test 2022 (cPRA complete)    44003.0     0.7276          0.7231      0.7925     0.7767   
                                             test 2023+ (all)            175006.0     0.7219          0.7164      0.7551     0.7524   
                                             test 2023+ (cPRA observed)   33008.0     0.6936          0.6880      0.7212     0.7193   

                                                                         AUC(t) 2y  
model                                        split                                  
Cox stratified A: strata dialysis            test 2022 (cPRA complete)      0.7653  
                                             test 2023+ (all)               0.7667  
                                             test 2023+ (cPRA observed)     0.7439  
Cox stratified B: strata dialysis + inactive test 2022 (cPRA complete)      0.7668  
                                             test 2023+ (all)               0.7697  
                                             test 2023+ (cPRA observed)     0.7478  
Cox binned, deceased-donor (unstratified)    test 2022 (cPRA complete)      0.7634  
                                             test 2023+ (all)               0.7645  
                                             test 2023+ (cPRA observed)     0.7364

In [20]:
# Paired bootstrap on the 2022 test set: stratified minus unstratified, both ranked by 2-year probability
risks = {"unstratified": risk_at(cph_nl_dd, data_nl_dd[is_test22])}
for name, (model, frame) in strat_models.items():
    risks[name] = risk_at(model, frame[is_test22])
draws = bootstrap_cindex(data_nl_dd[is_test22], risks)
diffs = pd.DataFrame({name: draws[name] - draws["unstratified"] for name in strat_models})
pd.DataFrame({
    "mean C": draws.mean(),
    "95% CI low": draws.quantile(0.025),
    "95% CI high": draws.quantile(0.975),
}).join(pd.DataFrame({
    "diff vs unstratified": diffs.mean(),
    "diff 95% CI low": diffs.quantile(0.025),
    "diff 95% CI high": diffs.quantile(0.975),
})).round(4)

,mean C,95% CI low,95% CI high,diff vs unstratified,diff 95% CI low,diff 95% CI high
unstratified,0.7278,0.7242,0.7316,NaN,NaN,NaN
A: strata dialysis,0.7286,0.7250,0.7324,0.0008,0.0003,0.0013
B: strata dialysis + inactive,0.7300,0.7262,0.7338,0.0022,0.0016,0.0029


In [21]:
# PH test after stratification (same 20k-row training sample as section 12)
model_b, frame_b = strat_models["B: strata dialysis + inactive"]
sample_b = frame_b.loc[sample.index]
cph_sample_b = CoxPHFitter(penalizer=0.01).fit(
    sample_b, duration_col="duration", event_col="event", strata=["dialysis_stratum", "inactive_stratum"]
)
ph_b = proportional_hazard_test(cph_sample_b, sample_b, time_transform="rank").summary
print(f"terms with p < 0.01: {(ph_b['p'] < 0.01).sum()} of {len(ph_b)}")
ph_b.sort_values("test_statistic", ascending=False).head(10).round(4)

terms with p < 0.01: 2 of 48


,test_statistic,p,-log2(p)
function_bin_<=40,58.7812,0.0000,45.6896
ABO_O,9.1144,0.0025,8.6232
age_bin_65-74,5.7675,0.0163,5.9367
age_bin_18-34,5.2440,0.0220,5.5049
function_bin_50-60,4.5150,0.0336,4.8955
ETHCAT_4,4.4116,0.0357,4.8081
ETHCAT_2,3.2820,0.0700,3.8356
ETHCAT_7,2.9412,0.0863,3.5337
REGION_9,2.7762,0.0957,3.3857
REGION_8,2.6915,0.1009,3.3092
